# 06 · Data cleaning: food helpline contacts (`Incoming Call.xlsx`)

**What it is:** the intake log of The Food Group's food and SNAP helpline. One row = one contact (phone call, voicemail, email, chat, text, or a clinic referral through the **SNAP Rx** program).

**Input:** `Data/Incoming Call.xlsx` (49,727 rows, Feb 2022 – Jul 2026) and the official county list.

**Outputs (`Data/processed/`):**
1. `calls_clean.csv`: one row per contact, with a clean county, reason group and referral flags.
2. `calls_county_month.csv`: contact counts for **every county × every month**, Jan 2023 – Jul 2026.

**Important limitation:** contacts are concentrated in the Twin Cities, largely because clinic and WIC partners there refer patients (SNAP Rx). Call volume reflects **where referral partners are** as well as where need is, so compare counties with care.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
OUT_DIR = DATA / "processed"
mmg = pd.read_csv(OUT_DIR / "mmg_mn_county_2022_2024.csv", dtype={"fips": str})
COUNTY_FIPS = dict(zip(mmg["county"], mmg["fips"]))      # official 87 counties -> FIPS
FIPS_COUNTY = {v: k for k, v in COUNTY_FIPS.items()}

raw = pd.read_excel(DATA / "Incoming Call.xlsx")
print("raw:", raw.shape, "| __ID unique:", raw["__ID"].is_unique)

raw: (49727, 31) | __ID unique: False


## Step 1: Columns

- **Kept:** contact ID, date, method, answered, length, caller county and region, household size, reason, SNAP status, SNAP Rx flag, referral destinations, referral source (partner) and how they heard.
- **Dropped:** system timestamps, follow-up workflow fields, best contact method, and `Call Description` (mostly operator notes such as "Left VM").

In [2]:
COLS = {"__ID": "call_id", "Date": "date", "CallMethod": "call_method", "Call Answered": "call_answered",
        "CallLength": "call_length", "County": "county_original", "Region": "region", "NumberInHouse": "household_size",
        "PrimaryReasonForCall": "reason_original", "SNAP Status": "snap_status", "isSNAPRx": "is_snap_rx",
        "ReferralInfoEntry": "referrals_raw", "Referral Source": "referral_source", "HowHear": "how_hear"}
calls = raw[list(COLS)].rename(columns=COLS)
calls["date"] = pd.to_datetime(calls["date"])
calls["is_snap_rx"] = calls["is_snap_rx"].eq(1) | calls["call_method"].eq("SNAP Rx")
print("contacts per year:", calls.groupby(calls["date"].dt.year).size().to_dict())

contacts per year: {2022: 41, 2023: 15434, 2024: 14600, 2025: 12590, 2026: 7062}


## Step 2: Duplicates

- **What:** remove rows that repeat the **same contact ID**; the repeats are identical apart from system timestamps.
- **Not removed:** about 280 rows that look identical but have **different** IDs. They are SNAP Rx clinic referrals entered with a date but no time, so several real patients referred on the same day from the same county look the same. They are kept.

In [3]:
before = len(calls)
calls = calls.drop_duplicates("call_id")
print(f"repeated contact IDs removed: {before - len(calls)} | rows: {len(calls)}")

repeated contact IDs removed: 2 | rows: 49725


## Step 3: Caller county → 87 counties + FIPS

- **What:**
  - Fix typos and variants: `Hennipen`, `hennHennepin`, `Sterns`, `Ramesy`, `rRoseau`, `Dakota County`, `Lake Of The Woods`, trailing spaces, …
  - Merge the city/suburb splits (same rules as the food shelf cleaning).
  - Set `county_status` to one of:
    - `mn_county`: matched one of the 87 counties;
    - `out_of_state`: `( Out of State )`;
    - `unknown`: blank, `(Unknown)`, `USA`, `h`, or ambiguous (`Cottonwood/Brown`).
- **Why:** FIPS joins to every other county dataset. Ambiguous values are not guessed.

In [4]:
TYPOS = {"Hennipen": "Hennepin", "hennHennepin": "Hennepin", "hennepin": "Hennepin", "Hennepin county": "Hennepin",
         "rasmey": "Ramsey", "Ramesy": "Ramsey", "rRoseau": "Roseau", "dakotaDakota": "Dakota", "dakota": "Dakota",
         "AAnoka": "Anoka", "Sterns": "Stearns", "FilmoreFillmore": "Fillmore", "St Louis": "St. Louis",
         "Lake Of The Woods": "Lake of the Woods"}


def clean_county(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip().str.replace(r"\s+County$", "", regex=True, case=False)
    s = s.replace(TYPOS)
    s = s.str.replace(r"-(Minneapolis|Suburbs|Saint Paul)$", "", regex=True)
    return s.replace({"Saint Louis": "St. Louis", "Lac Qui Parle": "Lac qui Parle", "Mcleod": "McLeod"})


calls["county"] = clean_county(calls["county_original"])
calls["fips"] = calls["county"].map(COUNTY_FIPS)
calls["county_status"] = np.select(
    [calls["fips"].notna(), calls["county_original"].astype("string").str.contains("Out of State").fillna(False).astype(bool)],
    ["mn_county", "out_of_state"], default="unknown")
calls.loc[calls["fips"].isna(), "county"] = None
print(calls["county_status"].value_counts().to_dict())
print("unmatched raw values sent to 'unknown':",
      sorted(calls.loc[(calls["county_status"] == "unknown") & calls["county_original"].notna(), "county_original"].unique()))
print("counties with at least one contact:", calls["fips"].nunique(), "of 87")

{'mn_county': 32749, 'unknown': 16268, 'out_of_state': 708}


unmatched raw values sent to 'unknown': ['(Unknown)', 'Cottonwood/Brown', 'USA', 'h']
counties with at least one contact: 87 of 87


## Step 4: Reason groups

- **What:** clean the 30+ reason labels (trailing spaces, accidentally merged labels) and group them:

| `reason_group` | Reasons |
|---|---|
| `snap_rx` | SNAP Rx (clinic referral for SNAP help) |
| `snap_help` | eligibility, application assistance / status, EBT card, recertification, interview, SNAP changes, E-SNAP, county office details, case-specific issues |
| `food_shelf` | Food Shelf |
| `other_food` | meal program, farmers market, affordable groceries, CSFP, NAPS, WIC, SUN Bucks, Meals on Wheels, damaged food |
| `other` | general info, other, non-food social services, retailer, medical assistance, map updates, grant, unknown / blank |

- **Why:** most questions need "food shelf requests" or "SNAP requests"; the full label is kept in `reason`.
- Merged labels (e.g. `Eligibility RequirementsSNAP Rx`) are assigned to their **first** label.

In [5]:
r = calls["reason_original"].astype("string").str.strip()
r = r.replace({"Eligibility RequirementsSNAP Rx": "Eligibility Requirements", "Application status": "Application Status",
               "County Office DetApplication Statusails": "County Office Details", "": pd.NA})
calls["reason"] = r
REASON_GROUP = {
    "SNAP Rx": "snap_rx", "Food Shelf": "food_shelf",
    **{k: "snap_help" for k in ["Eligibility Requirements", "Application Assistance", "Application Status", "EBT Card",
                                "Recertification", "Interview", "SNAP Changes", "E-SNAP", "County Office Details",
                                "Case-Specific Issues"]},
    **{k: "other_food" for k in ["Meal Program", "Farmers market", "Affordable Groceries", "CSFP", "NAPS", "WIC",
                                 "SUN Bucks", "Meals on Wheels", "Damaged Food"]},
}
calls["reason_group"] = calls["reason"].map(REASON_GROUP).fillna("other")
calls.groupby("reason_group")["reason"].value_counts(dropna=False).rename("contacts").to_frame()

contacts
reason_group reason                                  
food_shelf   Food Shelf                          8108
other        Other                               2679
             Unknown                             1043
             General Info                         829
             Other non-food social services       342
             Retailer                             140
             <NA>                                 100
             Map Updates                           14
             Medical Assistance                     7
             Grant                                  1
other_food   SUN Bucks                           1812
             Meal Program                         415
             Farmers market                       262
             Affordable Groceries                  74
             CSFP                                  34
             WIC                                   12
             NAPS                                   4
             Damaged Food                           1
             Meals on Wheels                        1
snap_help    Case-Specific Issues                4686
             Eligibility Requirements            3357
             EBT Card                            2450
             Application Status                  1603
             Application Assistance              1231
             Recertification                      427
             SNAP Changes                         392
             Interview                            215
             County Office Details                202
             E-SNAP                               192
snap_rx      SNAP Rx                            19092

## Step 5: Referral destinations → yes/no columns

- **What:** `referrals_raw` lists one or more destinations per contact (one per line). Turn the main ones into `ref_…` True/False columns, plus `n_referrals`.
- **Why:** e.g. "how many people were referred to a food shelf, per county per month" becomes a simple sum.

In [6]:
REF = {"ref_food_shelf": "Food Shelf", "ref_snap_info": "SNAP Info Given", "ref_mnbenefits": "MNBenefits",
       "ref_county_office": "County Office", "ref_meal_program": "Meal Program", "ref_fare_for_all": "Fare For All",
       "ref_211": "UW-211", "ref_ebt_service": "EBT Edge Service Center", "ref_farmers_market": "Farmers Market",
       "ref_sun_bucks": "SUN Bucks Hotline", "ref_market_bucks": "Market Bucks", "ref_wic": "WIC"}
items = calls["referrals_raw"].astype("string").str.split("\n")
items = items.map(lambda xs: [x.strip() for x in xs if x.strip()] if isinstance(xs, list) else [])
for col, label in REF.items():
    calls[col] = items.map(lambda xs, label=label: label in xs)
calls["n_referrals"] = items.map(len)
pd.Series({col: int(calls[col].sum()) for col in REF}, name="contacts referred").to_frame()

,contacts referred
ref_food_shelf,17194
ref_snap_info,13552
ref_mnbenefits,13189
ref_county_office,7882
ref_meal_program,4758
ref_fare_for_all,3417
ref_211,2965
ref_ebt_service,2675
ref_farmers_market,2402
ref_sun_bucks,1396


## Step 6: Save

- `calls_clean.csv`: every contact (2022 included). Only 41 contacts fall in 2022, so the log effectively **starts in January 2023**.
- `calls_county_month.csv`: **all 87 counties × every month from Jan 2023 to Jul 2026**. **Zero means no contact from that county was logged that month** (the log is the complete record of the helpline). Contacts without a Minnesota county are not in this table; their monthly counts are printed below.

In [7]:
calls["year"] = calls["date"].dt.year
calls["month"] = calls["date"].dt.to_period("M").dt.to_timestamp()
ORDER = (["call_id", "date", "year", "month", "county", "fips", "county_status", "county_original", "region",
          "call_method", "call_answered", "call_length", "household_size", "reason", "reason_group", "reason_original",
          "snap_status", "is_snap_rx"] + list(REF) + ["n_referrals", "referral_source", "how_hear"])
calls = calls[ORDER].drop(columns="reason_original").sort_values(["date", "call_id"]).reset_index(drop=True)
calls.to_csv(OUT_DIR / "calls_clean.csv", index=False)

mn = calls[(calls["county_status"] == "mn_county") & (calls["month"] >= "2023-01-01")]
agg = (mn.assign(one=1, food_shelf=mn["reason_group"].eq("food_shelf"), snap_rx=mn["reason_group"].eq("snap_rx"),
                 snap_help=mn["reason_group"].eq("snap_help"), other_food=mn["reason_group"].eq("other_food"))
         .groupby(["fips", "month"])
         .agg(contacts_total=("one", "sum"), contacts_food_shelf=("food_shelf", "sum"), contacts_snap_rx=("snap_rx", "sum"),
              contacts_snap_help=("snap_help", "sum"), contacts_other_food=("other_food", "sum"),
              referrals_food_shelf=("ref_food_shelf", "sum"), referrals_snap_application=("ref_mnbenefits", "sum")))
months = pd.date_range("2023-01-01", calls["month"].max(), freq="MS")
grid = pd.MultiIndex.from_product([sorted(COUNTY_FIPS.values()), months], names=["fips", "month"])
county_month = agg.reindex(grid, fill_value=0).reset_index()
county_month.insert(1, "county", county_month["fips"].map(FIPS_COUNTY))
county_month.to_csv(OUT_DIR / "calls_county_month.csv", index=False)
print("saved calls_clean.csv:", calls.shape, "| calls_county_month.csv:", county_month.shape,
      f"({county_month['fips'].nunique()} counties × {len(months)} months)")
print("share of 2023+ contacts with a Minnesota county:",
      f"{len(mn) / (calls['month'] >= '2023-01-01').sum():.1%}")
print("share of county-attributed contacts from the 7-county Twin Cities metro:",
      f"{mn['county'].isin(['Anoka', 'Carver', 'Dakota', 'Hennepin', 'Ramsey', 'Scott', 'Washington']).mean():.1%}")

saved calls_clean.csv: (49725, 32) | calls_county_month.csv: (3741, 10) (87 counties × 43 months)
share of 2023+ contacts with a Minnesota county: 65.9%
share of county-attributed contacts from the 7-county Twin Cities metro: 87.2%


## Summary

- **`calls_clean.csv`:** 49,725 contacts (2 repeated IDs removed), with a clean county, reason group and 12 referral yes/no columns.
- **Caller county:** 32,749 contacts matched to a Minnesota county, 708 out of state, 16,268 unknown (mostly blank). Every one of the 87 counties has at least one contact.
- **`calls_county_month.csv`:** all 87 counties × 43 months (Jan 2023 – Jul 2026) = 3,741 rows. Zero means no logged contact.
- **Main limitation:** 87% of county-attributed contacts come from the 7-county Twin Cities metro, largely clinic and WIC referral partners (SNAP Rx). Compare counties per resident and with care, and keep SNAP Rx separate (`contacts_snap_rx`) from people contacting the helpline themselves.
- **Starts in 2023:** only 41 contacts in 2022, so analyses should start in January 2023.